# Multilayer CNN

In [3]:
import tensorflow_datasets as tfds
import tensorflow as tf

In [ ]:
# loading the data
mnist_bldr = tfds.builder('mnist')
mnist_bldr.download_and_prepare()
# shuffel false, beacuse we will split the training set into two parts: smaller training dataset and a validation dataset
# if shuffling was on, it would incur resuffling of the dataset each time we fetched a mini-batch of data
# can cause false performance estimation of the model
datasets = mnist_bldr.as_dataset(shuffle_files=False)
mnist_train_orig = datasets['train']
mnist_test_orig = datasets['test']

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

I0000 00:00:1790768423.949213   26234 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790768423.951431   26234 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790768424.224836   26234 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790768425.512719   26234 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

...

E0000 00:00:1790768426.786804   25935 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Shuffling /home/skramstad/tensorflow_datasets/mnist/incomplete.12FI7O_3.0.1/mnist-train.tfrecord-[0-9][0-9][0-…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /home/skramstad/tensorflow_datasets/mnist/incomplete.12FI7O_3.0.1/mnist-test.tfrecord-[0-9][0-9][0-9…

Dataset mnist downloaded and prepared to /home/skramstad/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.


In [5]:
# preparing the dataset
BUFFER_SIZE = 10000
BATCH_SIZE = 64
NUM_EPOCHS = 20

mnist_train = mnist_train_orig.map(
    lambda item: (tf.cast(item['image'], tf.float32) / 255.0,
                  tf.cast(item['label'], tf.int32))
)

mnist_test = mnist_test_orig.map(
    lambda item: (tf.cast(item['image'], tf.float32) / 255.0,
                  tf.cast(item['label'], tf.int32))
)

tf.random.set_seed(1)

mnist_train = mnist_train.shuffle(buffer_size=BUFFER_SIZE,
                                  reshuffle_each_iteration = False)


mnist_valid = mnist_train.take(10000).batch(BATCH_SIZE)
mnist_train = mnist_train.skip(10000).batch(BATCH_SIZE)

In [ ]:
# constructing the cnn
model = tf.keras.Sequential()
model.add(tf.keras.layers.Conv2D(
    filters=32, kernel_size=(5,5),
    strides=(1,1), padding='same',
    data_format='channels_last',
    name='conv_1', activation='relu')
)

model.add(tf.keras.layers.MaxPool2D(
    pool_size=(2,2), name='pool_1'
))